In [7]:
import sys

print(sys.executable)

/usr/local/bin/python3.11


In [8]:
import sys
!{sys.executable} -m pip install -U huggingface_hub

  Using cached huggingface_hub-1.16.1-py3-none-any.whl.metadata (14 kB)
  Using cached hf_xet-1.5.0-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached click-8.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
Using cached huggingface_hub-1.16.1-py3-none-any.whl (668 kB)
Using cached hf_xet-1.5.0-cp37-abi3-macosx_11_0_arm64.whl (3.8 MB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached typer-0.25.1-py3-none-any.whl (58 kB)
Using cached annotated_doc-0.0.4-py3-none-any.whl (5.3 kB)
Using cached click-8.4.1-py3-none-any.whl (116 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
  Attempting uninstall: h

In [1]:
#
from huggingface_hub import login

login()

In [2]:
# %%
from pathlib import Path
import json

UPLOAD_DIR = Path("/Users/omarleosamman/Downloads/hf_xgb_candidate58_upload")

print("Files in upload folder:")
for file in UPLOAD_DIR.iterdir():
    print("-", file.name)

Files in upload folder:
- xgb_multiclass_candidate58_test_predictions_with_hcp_id.csv
- xgb_multiclass_test_predictions.csv
- xgb_multiclass_candidate58_test_predictions.csv
- best_xgb_multiclass_atseg_candidate58.joblib
- xgb_multiclass_candidate58_metadata.json


In [3]:
# %%
metadata_file = UPLOAD_DIR / "xgb_multiclass_candidate58_metadata.json"

with open(metadata_file, "r", encoding="utf-8") as f:
    metadata = json.load(f)

metadata

{'model': 'XGBClassifier',
 'task': 'SEG_A vs SEG_B vs SEG_C',
 'selection_reason': 'Candidate 58 selected due to better SEG_C recall while maintaining competitive Macro F1.',
 'best_params': {'subsample': 0.9,
  'reg_lambda': 1,
  'reg_alpha': 0.1,
  'n_estimators': 200,
  'min_child_weight': 1,
  'max_depth': 3,
  'learning_rate': 0.03,
  'gamma': 0.05,
  'colsample_bytree': 0.75},
 'decision_rule': 'argmax over predicted class probabilities',
 'label_mapping': {'0': 'SEG_A', '1': 'SEG_B', '2': 'SEG_C'},
 'test_metrics': {'accuracy': 0.6449579831932774,
  'macro_f1': 0.5792501739473731,
  'weighted_f1': 0.6479920763036838,
  'SEG_A_precision': 0.7828525641025641,
  'SEG_A_recall': 0.7626854020296643,
  'SEG_A_f1': 0.7726374060893634,
  'SEG_B_precision': 0.5787878787878787,
  'SEG_B_recall': 0.5701492537313433,
  'SEG_B_f1': 0.5744360902255639,
  'SEG_C_precision': 0.3728813559322034,
  'SEG_C_recall': 0.41025641025641024,
  'SEG_C_f1': 0.390677025527192,
  'bc_recall_avg': 0.4902028

In [4]:
# %%
test_metrics = metadata.get("test_metrics", {})
best_params = metadata.get("best_params", {})
model_name = metadata.get("model", "XGBClassifier")
selected_candidate = metadata.get("selected_candidate", "58")

readme_text = f'''
---
license: other
tags:
- multiclass-classification
- healthcare
- physician-segmentation
- xgboost
---

# XGBoost Multiclass ATSEG Classifier - Candidate {selected_candidate}

This repository contains the selected XGBoost multiclass model for direct ATSEG classification.

## Task

Multiclass classification:

- `0`: SEG_A
- `1`: SEG_B
- `2`: SEG_C

## Selected Model

Model: `{model_name}`  
Selected candidate: `{selected_candidate}`

Candidate {selected_candidate} was selected because it improved SEG_C recall while maintaining competitive Macro F1.

## Best Parameters

{best_params}

## Test Metrics

- Accuracy: {test_metrics.get("accuracy", "NA")}
- Macro F1: {test_metrics.get("macro_f1", "NA")}
- Weighted F1: {test_metrics.get("weighted_f1", "NA")}
- SEG_A Recall: {test_metrics.get("SEG_A_recall", "NA")}
- SEG_B Recall: {test_metrics.get("SEG_B_recall", "NA")}
- SEG_C Recall: {test_metrics.get("SEG_C_recall", "NA")}

## Files

- `best_xgb_multiclass_candidate58.joblib`: trained XGBoost multiclass model
- `xgb_multiclass_candidate58_metadata.json`: model parameters, label mapping and metrics
- `xgb_candidate58_test_predictions.csv`: test predictions with predicted probabilities
- `xgb_multiclass_hyperparameter_search_results.csv`: hyperparameter search results

## Reuse

    import joblib
    import json
    import numpy as np

    model = joblib.load("best_xgb_multiclass_candidate58.joblib")

    with open("xgb_multiclass_candidate58_metadata.json", "r") as f:
        metadata = json.load(f)

    proba = model.predict_proba(X_new_flat)
    pred_encoded = np.argmax(proba, axis=1)

    label_mapping = metadata["label_mapping"]
    pred_label = [label_mapping[str(i)] for i in pred_encoded]

## Input Format

The model expects flattened temporal tensor features with the same feature order used during training.
'''

with open(UPLOAD_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_text)

print("README created:", UPLOAD_DIR / "README.md")

README created: /Users/omarleosamman/Downloads/hf_xgb_candidate58_upload/README.md


In [7]:
# %%
from huggingface_hub import create_repo, upload_folder

HF_USERNAME = "omarleosamman"  # cambia esto
REPO_NAME = "xgb-multiclass-atseg-candidate58"
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True
)

upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=str(UPLOAD_DIR),
    commit_message="Upload selected XGBoost multiclass candidate 58"
)

print(f"Uploaded to: https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded to: https://huggingface.co/omarleosamman/xgb-multiclass-atseg-candidate58
